<div dir="rtl" align="right">

# مختبر الأسبوع الثاني · اليوم الرابع
## متى يفشل النموذج؟ الإفراط في التعلّم

بالأمس بنينا نموذجًا يتنبّأ باستهلاك الوقود. واليوم نسأل سؤالًا أعمق: **متى نثق بنموذجنا؟ ومتى يخدعنا؟**

سنكتشف أخطر فخٍّ في تعلّم الآلة: نموذجٌ يبدو ممتازًا... وهو في الحقيقة عديم الفائدة.

**البيانات:** `cars.csv` نفسها التي تعرفها من الأمس.


</div>

<div dir="rtl" align="right">

### الخطوة 1 — حمّل ونظّف وقسّم
مراجعةٌ سريعة لخطوات الأمس، في خليةٍ واحدة.

</div>

In [ ]:
%pip install -q scikit-learn matplotlib

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# البيانات تُحمَّل من الإنترنت مباشرة، لا حاجة لرفع أي ملف
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
try:
    from pyodide.http import open_url
    df = pd.read_csv(open_url(url)).dropna()
except ImportError:
    df = pd.read_csv(url).dropna()

X = df[["weight"]]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

print("Training cars:", len(X_train))
print("Test cars:", len(X_test))

<div dir="rtl" align="right">

### الخطوة 2 — نموذجٌ يبدو مثاليًّا
سندرّب شجرة قرارٍ بلا أي قيد، ونقيس خطأها على **بيانات التدريب**.

› انظر إلى الرقم. هل النموذج ممتاز؟

</div>

In [ ]:
deep_model = DecisionTreeRegressor(random_state=42)
deep_model.fit(X_train, y_train)

train_error = mean_absolute_error(y_train, deep_model.predict(X_train))
print("Error on TRAINING data:", train_error)

<div dir="rtl" align="right">

### الخطوة 3 — الحقيقة الصادمة
الآن نقيس النموذج نفسه على **بيانات الاختبار** — سيّاراتٌ لم يرها أبدًا.

</div>

In [ ]:
test_error = mean_absolute_error(y_test, deep_model.predict(X_test))

print("Error on TRAINING data:", train_error)
print("Error on TEST data:    ", test_error)

<div dir="rtl" align="right">

### الخطوة 4 — ماذا حدث؟
النموذج **حفظ** بيانات التدريب بدل أن **يتعلّم** النمط. وهذا اسمه **الإفراط في التعلّم**.

مثاله: طالبٌ حفظ إجابات الواجب حرفيًّا. سيحصل على 100% في الواجب، لكنه سيفشل في الامتحان.

› الحلّ: نمنع النموذج من الحفظ، بأن نحدّد له عمقًا أقصى.

</div>

In [ ]:
simple_model = DecisionTreeRegressor(max_depth=1, random_state=42)
simple_model.fit(X_train, y_train)

print("Simple model - train:", mean_absolute_error(y_train, simple_model.predict(X_train)))
print("Simple model - test: ", mean_absolute_error(y_test, simple_model.predict(X_test)))

<div dir="rtl" align="right">

### الخطوة 5 — جرّب كل الأعماق
النموذج العميق يحفظ، والبسيط جدًّا لا يتعلّم كفاية. فأين النقطة المثالية؟ لنجرّب.

› راقب العمودين: خطأ التدريب يهبط دائمًا، لكن ماذا يفعل خطأ الاختبار؟

</div>

In [ ]:
depths = []
train_errors = []
test_errors = []

for d in range(1, 13):
    model = DecisionTreeRegressor(max_depth=d, random_state=42)
    model.fit(X_train, y_train)

    tr = mean_absolute_error(y_train, model.predict(X_train))
    te = mean_absolute_error(y_test, model.predict(X_test))

    depths.append(d)
    train_errors.append(tr)
    test_errors.append(te)

    print("depth", d, "-> train", round(tr, 2), "| test", round(te, 2))

<div dir="rtl" align="right">

### الخطوة 6 — ارسم القصّة
المخطّط يحكي كل شيء: الخطّ الأزرق يهبط دائمًا، والأحمر ينزل ثم **يصعد**.

› أين أدنى نقطةٍ في الخطّ الأحمر؟ تلك هي أفضل نموذج.

</div>

In [ ]:
import matplotlib.pyplot as plt

plt.plot(depths, train_errors, label="train")
plt.plot(depths, test_errors, color="red", label="test")
plt.xlabel("max_depth")
plt.ylabel("error (MAE)")
plt.legend()
plt.show()

<div dir="rtl" align="right">

### الخطوة 7 — اختر الأفضل
نختار العمق صاحب أقلّ خطأ اختبار — لا أقلّ خطأ تدريب!

</div>

In [ ]:
best_depth = depths[test_errors.index(min(test_errors))]
print("Best depth:", best_depth)
print("Its test error:", min(test_errors))

best_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_model.fit(X_train, y_train)

<div dir="rtl" align="right">

### الخطوة 8 — حسّن النموذج بميزاتٍ أكثر
طريقةٌ أخرى للتحسين: أعطِ النموذج معلوماتٍ أكثر عن كل سيّارة.

</div>

In [ ]:
X4 = df[["weight", "horsepower", "cylinders", "model_year"]]

X4_train, X4_test, y_train, y_test = train_test_split(
    X4, y, test_size=0.3, random_state=42)

model4 = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
model4.fit(X4_train, y_train)

print("1 feature  - test error:", min(test_errors))
print("4 features - test error:", mean_absolute_error(y_test, model4.predict(X4_test)))

<div dir="rtl" align="right">

## التحدّي

1. في الخطوة 5، ما أكبر فرقٍ رأيته بين خطأ التدريب وخطأ الاختبار؟ عند أيّ عمق؟
2. أعِد الخطوة 5 مع الميزات الأربع. هل يتغيّر العمق الأفضل؟
3. غيّر `test_size` إلى `0.5`. هل يبقى العمق الأفضل نفسه؟

اكتب حلّك في الخلية التالية.

</div>

In [ ]:
# your answer here
